In [5]:
"""
scanner.py — Analyze your real equations

Scans metadata.csv to extract:
- Operator frequency → weights for synthetic generation
- Variable names used
- Equation length distribution
- Variables-per-equation distribution
- Tree depth estimates

Run standalone: python scanner.py --meta data/metadata.csv
"""

import os, csv, argparse
import numpy as np
from collections import Counter
from operators import OPERATORS


def scan(meta_csv):
    """
    Scan real equations. Returns dict with all distributions.
    
    Finds variables from THREE sources:
    1. The 'variables' column in metadata (e.g. "['theta', 'theta1']")
    2. Tokens in prefix expression that aren't operators or numbers
    3. Data file column headers (if accessible)
    """
    op_counts = Counter()
    var_names = set()           # all unique variables across dataset
    var_per_eq = {}             # per-equation variable lists
    eq_lengths = []
    n_vars_list = []
    const_counts = []
    samples = []

    with open(meta_csv) as f:
        for row in csv.DictReader(f):
            prefix = row['expression_prefix'].strip().split()
            eq_lengths.append(len(prefix))
            fname = row['filename']

            eq_vars = set()
            n_consts = 0

            # --- Source 1: 'variables' column ---
            if 'variables' in row and row['variables']:
                try:
                    declared_vars = eval(row['variables'])  # e.g. "['theta','theta1']"
                    if isinstance(declared_vars, list):
                        for v in declared_vars:
                            v = str(v).strip()
                            var_names.add(v)
                            eq_vars.add(v)
                except:
                    pass

            # --- Source 2: tokens in prefix expression ---
            for t in prefix:
                if t in OPERATORS:
                    op_counts[t] += 1
                elif t in ("pi", "e"):
                    n_consts += 1
                else:
                    try:
                        float(t)
                        n_consts += 1
                    except ValueError:
                        # not an operator, not a number, not pi/e → variable
                        var_names.add(t)
                        eq_vars.add(t)

            n_vars_list.append(len(eq_vars))
            const_counts.append(n_consts)
            var_per_eq[fname] = sorted(eq_vars)

            samples.append({
                'filename': fname,
                'prefix': prefix,
                'variables': sorted(eq_vars),
                'n_vars': len(eq_vars),
                'n_consts': n_consts,
                'length': len(prefix),
            })

    # --- Source 3: scan data file headers ---
    data_dir = os.path.dirname(meta_csv) or "."
    header_vars = set()
    for s in samples[:10]:  # check first 10 files
        fpath = os.path.join(data_dir, s['filename'] + '.csv')
        if os.path.exists(fpath):
            try:
                with open(fpath) as df:
                    headers = df.readline().strip().split(',')
                    for h in headers:
                        h = h.strip()
                        if h and h != 'y' and h not in ('',):
                            header_vars.add(h)
            except:
                pass

    # merge header vars into main set
    new_from_headers = header_vars - var_names
    if new_from_headers:
        var_names |= new_from_headers

    # derive weights from counts
    total = sum(op_counts.values()) or 1
    weights = {}
    for op in OPERATORS:
        if op in op_counts:
            weights[op] = max(1, int(100 * op_counts[op] / total))
        else:
            weights[op] = OPERATORS[op]["weight"]

    # variable frequency (how often each var appears)
    var_freq = Counter()
    for s in samples:
        for v in s['variables']:
            var_freq[v] += 1

    result = {
        'n_equations': len(samples),
        'variables': sorted(var_names),
        'variable_frequency': dict(var_freq.most_common()),
        'variables_from_headers': sorted(new_from_headers) if new_from_headers else [],
        'var_per_equation': var_per_eq,
        'op_counts': dict(op_counts.most_common()),
        'op_weights': weights,
        'n_vars_distribution': n_vars_list,
        'eq_length_distribution': eq_lengths,
        'const_count_distribution': const_counts,
        'samples': samples,
    }
    return result


def print_report(stats):
    """Pretty print the analysis."""
    print(f"\n{'='*50}")
    print(f"REAL DATA ANALYSIS")
    print(f"{'='*50}")
    print(f"Total equations: {stats['n_equations']}")

    print(f"\nVariables found ({len(stats['variables'])}):")
    print(f"  {stats['variables']}")

    if stats.get('variables_from_headers'):
        print(f"\n  Extra vars found in CSV headers (not in prefix):")
        print(f"  {stats['variables_from_headers']}")

    # variable frequency
    print(f"\nVariable frequency (how many equations use each):")
    for var, count in stats['variable_frequency'].items():
        bar = "█" * (count * 30 // max(stats['variable_frequency'].values()))
        print(f"  {var:12s}: {bar} ({count})")

    nvd = stats['n_vars_distribution']
    print(f"\nVars per equation: min={min(nvd)} max={max(nvd)} avg={np.mean(nvd):.1f}")
    vc = Counter(nvd)
    for v in sorted(vc):
        bar = "█" * (vc[v] * 30 // max(vc.values()))
        print(f"  {v} vars: {bar} ({vc[v]})")

    eld = stats['eq_length_distribution']
    print(f"\nEquation length: min={min(eld)} max={max(eld)} avg={np.mean(eld):.1f}")

    ccd = stats['const_count_distribution']
    print(f"Constants per eq: min={min(ccd)} max={max(ccd)} avg={np.mean(ccd):.1f}")

    print(f"\nOperator counts:")
    for op, count in stats['op_counts'].items():
        bar = "█" * (count * 30 // max(stats['op_counts'].values()))
        print(f"  {op:6s}: {bar} ({count})")

    print(f"\nDerived weights (for synthetic generation):")
    for op, w in stats['op_weights'].items():
        print(f"  {op:6s}: {w}")

    # show a few per-equation examples
    print(f"\nSample equations with their variables:")
    for s in stats['samples'][:5]:
        print(f"  {s['filename']:12s} vars={s['variables']}  "
              f"len={s['length']}  consts={s['n_consts']}")



stats = scan(r"C:\Users\Yajat\Documents\SymbolicRegression - LLM-Jepa\preprocessed_data\feynman_preprocessed.csv")
print_report(stats)


REAL DATA ANALYSIS
Total equations: 100

Variables found (96):
  ['A', 'A_vec', 'B', 'Bx', 'By', 'Bz', 'C', 'E_n', 'Ef', 'F', 'G', 'H', 'I', 'I1', 'I2', 'I_0', 'Int_0', 'Jz', 'M', 'Nn', 'Pwr', 'T', 'T1', 'T2', 'U', 'V', 'V1', 'V2', 'Volt', 'Y', 'a', 'alpha', 'arcsin', 'beta', 'c', 'chi', 'd', 'd1', 'd2', 'delta', 'epsilon', 'g', 'g_', 'gamma', 'h', 'k', 'k_spring', 'kappa', 'kb', 'lambd', 'ln', 'm', 'm1', 'm2', 'm_0', 'mob', 'mom', 'mu', 'mu_drift', 'n', 'n_0', 'n_rho', 'omega', 'omega_0', 'p', 'p_d', 'pr', 'q', 'q1', 'q2', 'r', 'r1', 'r2', 'rho', 'rho_c_0', 'sigma', 'sigma_den', 't', 'tanh', 'theta', 'theta1', 'theta2', 'u', 'v', 'w', 'x', 'x1', 'x2', 'x3', 'y', 'y1', 'y2', 'y3', 'z', 'z1', 'z2']

Variable frequency (how many equations use each):
  c           : ██████████████████████████████ (19)
  epsilon     : ██████████████████████████████ (19)
  q           : ██████████████████████████ (17)
  v           : █████████████████████████ (16)
  r           : ██████████████████████ (14